# BurVision Dataset Builder: 140+ Myanmar Unicode Fonts & Native Browser HarfBuzz Engine

This notebook builds a high-resolution, layout-aware synthetic Burmese OCR dataset (`marventun/burvision-dataset`) with image and text annotations:

| Feature | Specification |
| :--- | :--- |
| **Total Myanmar Unicode Fonts** | **140+ Verified Myanmar Unicode Fonts** |
| **Intra-Page Font Mixing** | **Multi-font mixing on the same page** (Distinct Header Font + Body Font) |
| **Structural Ground Truth** | **Full Markdown Hierarchy** (`# H1`, `## H2`, `### H3`, `**bold**`, `*italic*`, `<u>underline</u>`) |
| **Shaping Engine** | **Native Browser HarfBuzz (`mym2`) + DirectWrite** (100% proper `ေ`, `ြ`, and stacked consonants) |
| **Physical Scan Realism** | **Open-book spine shadow, aged paper ivory/parchment, grain & skew** |

### Multi-Source Font Harvester (140+ Unique Unicode Fonts):
Cell 3 automatically harvests and deduplicates `.ttf` files from **4 open-source repositories**:
1. [`AungMyoKyaw/Myanmar-Unicode-Fonts`](https://github.com/AungMyoKyaw/Myanmar-Unicode-Fonts) (78 fonts)
2. [`Google Fonts (Noto Sans & Noto Serif Myanmar Full Weight Spectrum)`](https://github.com/google/fonts) (16 weights from Thin 100 to Black 900)
3. [`lecramyajiv/fonts-myanmar-extra`](https://github.com/lecramyajiv/fonts-myanmar-extra) (Debian/Slackware extra collection: *Masterpiece Spring Rev*, *Masterpiece Lakwel*, *Masterpiece CTL*, *Masterpiece Stadium*, *Masterpiece TawWin*, *Masterpiece UniHand*, *Masterpiece UniSans*, *Masterpiece UniSerif*, *Masterpiece UniType*, *Masterpiece YayChanzin*, *Myanmar Chatu*, *Myanmar ChatuLight*, *Myanmar HeadOne*, *Myanmar Khyay*, *Myanmar Sanpya*, *Myanmar Waso*, *Kayphodu*)
4. [`saturngod/myanmar-unicode-fonts`](https://github.com/saturngod/myanmar-unicode-fonts) (Community designer Burmese Unicode fonts)

## 1. Install Required Packages
Installs `selenium` (uses your installed Chrome or Microsoft Edge automatically), `pillow`, `numpy`, `tqdm`, and `paraencoder`.

In [ ]:
%pip install -q selenium pillow numpy tqdm paraencoder

## 2. Initialize Project Directories and Settings
Set the project folder, page dimensions, text chunk size, validation split, random seed, and archive name here. The setup creates the working directories and cleans out any non-Unicode legacy fonts (`Zawgyi-One`, `WININNWA`).

In [ ]:
import os

PROJECT_ROOT = "burmese_ocr_project"
TARGET_CHARS_PER_PAGE = 1050
VALIDATION_FRACTION = 0.10
RANDOM_SEED = 42
PAGE_WIDTH_PX = 794
PAGE_HEIGHT_PX = 1123
ZIP_NAME = "burmese_ocr_poc"

RAW_TEXTS_DIR = os.path.join(PROJECT_ROOT, "raw_texts")
FONTS_DIR = os.path.join(PROJECT_ROOT, "fonts")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "dataset_output")
IMAGES_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "images")

for d in [RAW_TEXTS_DIR, FONTS_DIR, IMAGES_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Ensure legacy non-Unicode fonts are removed
for bad_font in ["Zawgyi-One.ttf", "WININNWA.TTF", "Padauk-Zawgyi.ttf", "PadaukGhost.ttf"]:
    p = os.path.join(FONTS_DIR, bad_font)
    if os.path.exists(p):
        try:
            os.remove(p)
            print(f"Removed legacy non-Unicode font: {bad_font}")
        except Exception:
            pass

print("Directories initialized:")
print(f"  ├── {RAW_TEXTS_DIR}     <- Place your UTF-8 .txt files here")
print(f"  ├── {FONTS_DIR}         <- Downloaded Myanmar Unicode fonts stored here")
print(f"  └── {OUTPUT_DIR}    <- Final exported dataset (images + train.jsonl + val.jsonl)")

## 3. Multi-Source Font Harvester (140+ Free Myanmar Unicode Fonts)
Downloads the 78 fonts from `AungMyoKyaw/Myanmar-Unicode-Fonts`, 16 weights from `Google Fonts`, and automatically harvests all extra Unicode `.ttf` fonts from `lecramyajiv/fonts-myanmar-extra` and `saturngod/myanmar-unicode-fonts` via GitHub ZIP archives.

In [ ]:
import urllib.request
import zipfile
import io
from tqdm import tqdm

GH_BASE = "https://raw.githubusercontent.com/AungMyoKyaw/Myanmar-Unicode-Fonts/master/"
GF_BASE = "https://raw.githubusercontent.com/google/fonts/main/ofl/"

DIRECT_FONT_LIST = [
    ("Cherry_Unicode.ttf", GH_BASE + "CherryUnicode/Cherry_Unicode.ttf"),
    ("Jasmine_Unicode.ttf", GH_BASE + "JasmineUnicode/Jasmine_Unicode.ttf"),
    ("KT_02.ttf", GH_BASE + "KT_Unicode/KT_02.ttf"),
    ("KT_03.ttf", GH_BASE + "KT_Unicode/KT_03.ttf"),
    ("MON3Anonta1.ttf", GH_BASE + "MON3_Anonta/MON3Anonta1.ttf"),
    ("MasterpieceUniRound.ttf", GH_BASE + "MasterpieceUniRound/MasterpieceUniRound.ttf"),
    ("mm3-multi-os.ttf", GH_BASE + "Myanmar3/mm3-multi-os.ttf"),
    ("mm3_05-Jan-2011.ttf", GH_BASE + "Myanmar3/mm3_05-Jan-2011.ttf"),
    ("mm3h.ttf", GH_BASE + "Myanmar3/mm3h.ttf"),
    ("mm3sh.ttf", GH_BASE + "Myanmar3/mm3sh.ttf"),
    ("MM3_Paoh.ttf", GH_BASE + "Myanmar3_PaOh/MM3_Paoh.ttf"),
    ("MyanmarAngoun.ttf", GH_BASE + "MyanmarAngoun/MyanmarAngoun.ttf"),
    ("MyanmarBlack.ttf", GH_BASE + "MyanmarBlack/MyanmarBlack.ttf"),
    ("myanmarcensus.ttf", GH_BASE + "MyanmarCensus/myanmarcensus.ttf"),
    ("MyanmarGantGaw.ttf", GH_BASE + "MyanmarGantGaw/MyanmarGantGaw.ttf"),
    ("MyanmarKhway.ttf", GH_BASE + "MyanmarKhway/MyanmarKhway.ttf"),
    ("MyanmarKuttar.ttf", GH_BASE + "MyanmarKuttar/MyanmarKuttar.ttf"),
    ("MyanmarNayone.ttf", GH_BASE + "MyanmarNayone/MyanmarNayone.ttf"),
    ("MyanmarNjaun.ttf", GH_BASE + "MyanmarNjaun/MyanmarNjaun.ttf"),
    ("MyanmarPaOhOne.ttf", GH_BASE + "MyanmarPaOhOne/MyanmarPaOhOne.ttf"),
    ("MIUI_Unicode.ttf", GH_BASE + "MyanmarPaOhRosemary/MIUI%20Unicode.ttf"),
    ("MyanmarPauklay.ttf", GH_BASE + "MyanmarPauklay/MyanmarPauklay.ttf"),
    ("MyanmarPhetsot.ttf", GH_BASE + "MyanmarPhetsot/MyanmarPhetsot.ttf"),
    ("MyanmarPixel.ttf", GH_BASE + "MyanmarPixel/MyanmarPixel.ttf"),
    ("MyanmarPixelSmooth.ttf", GH_BASE + "MyanmarPixel/MyanmarPixelSmooth.ttf"),
    ("MyanmarPonenyet.ttf", GH_BASE + "MyanmarPonenyet/MyanmarPonenyet.ttf"),
    ("Myanmar_Punk.ttf", GH_BASE + "MyanmarPunk/Myanmar_Punk.ttf"),
    ("Myanmar_Rosemarry.ttf", GH_BASE + "MyanmarRosemarry/Myanmar_Rosemarry.ttf"),
    ("MyanmarSabae.ttf", GH_BASE + "MyanmarSabae/MyanmarSabae.ttf"),
    ("MyanmarSagar.ttf", GH_BASE + "MyanmarSagar/MyanmarSagar.ttf"),
    ("MyanmarSansPro.ttf", GH_BASE + "MyanmarSansPro/MyanmarSansPro.ttf"),
    ("MyanmarSquare.ttf", GH_BASE + "MyanmarSquare/MyanmarSquare.ttf"),
    ("MyanmarSquareLight.ttf", GH_BASE + "MyanmarSquare/MyanmarSquareLight.ttf"),
    ("MyanmarTagu.ttf", GH_BASE + "MyanmarTagu/MyanmarTagu.ttf"),
    ("MyanmarTaungyi.ttf", GH_BASE + "MyanmarTaungyi/MyanmarTaungyi.ttf"),
    ("mmrtext.ttf", GH_BASE + "MyanmarText/mmrtext.ttf"),
    ("mmrtextb.ttf", GH_BASE + "MyanmarText/mmrtextb.ttf"),
    ("MyanmarTextMelano.ttf", GH_BASE + "MyanmarTextMelano/MyanmarTextMelano.ttf"),
    ("MyanmarThuriya.ttf", GH_BASE + "MyanmarThuriya/MyanmarThuriya.ttf"),
    ("MyanmarThuriya_volt.ttf", GH_BASE + "MyanmarThuriya/MyanmarThuriya_volt.ttf"),
    ("MyanmarYinmar.ttf", GH_BASE + "MyanmarYinmar/MyanmarYinmar.ttf"),
    ("NK_Grand.ttf", GH_BASE + "Namkhon/NK%20Grand.ttf"),
    ("NK_SSmart2.ttf", GH_BASE + "Namkhon/NK_SSmart2.ttf"),
    ("NK_SSmart3.ttf", GH_BASE + "Namkhon/NK_SSmart3.ttf"),
    ("NK_SSmart4.ttf", GH_BASE + "Namkhon/NK_SSmart4.ttf"),
    ("NamKhoneUnicode.ttf", GH_BASE + "Namkhon/NamKhoneUnicode.ttf"),
    ("Namkhon_Web_pro.ttf", GH_BASE + "Namkhon/Namkhon_Web_pro.ttf"),
    ("NotoSansMyanmar-Bold.ttf", GH_BASE + "NotoSansMyanmar/NotoSansMyanmar-Bold.ttf"),
    ("NotoSansMyanmar-Regular.ttf", GH_BASE + "NotoSansMyanmar/NotoSansMyanmar-Regular.ttf"),
    ("NotoSansMyanmarUI-Bold.ttf", GH_BASE + "NotoSansMyanmar/NotoSansMyanmarUI-Bold.ttf"),
    ("NotoSansMyanmarUI-Regular.ttf", GH_BASE + "NotoSansMyanmar/NotoSansMyanmarUI-Regular.ttf"),
    ("Notosan_Myanmar_UI_Bold.ttf", GH_BASE + "NotoSansMyanmar/Notosan%20Myanmar%20UI%20Bold.ttf"),
    ("Padauk-bold.ttf", GH_BASE + "Padauk/Padauk-bold.ttf"),
    ("Padauk.ttf", GH_BASE + "Padauk/Padauk.ttf"),
    ("padauk-book.ttf", GH_BASE + "Padauk/padauk-book.ttf"),
    ("padauk-bookbold.ttf", GH_BASE + "Padauk/padauk-bookbold.ttf"),
    ("PadaukGrandPro_v2.ttf", GH_BASE + "PadaukGrandPro/PadaukGrandPro%20v2.ttf"),
    ("Padauk_Kyaungchikote.ttf", GH_BASE + "PadaukKyaungchikote/Padauk%20Kyaungchikote.ttf"),
    ("PadaukSagar.ttf", GH_BASE + "PadaukSagar/PadaukSagar.ttf"),
    ("PadaukSgaw-Bold.ttf", GH_BASE + "PadaukSgaw/PadaukSgaw-Bold.ttf"),
    ("PadaukSgaw-book.ttf", GH_BASE + "PadaukSgaw/PadaukSgaw-book.ttf"),
    ("PadaukSgaw-bookbold.ttf", GH_BASE + "PadaukSgaw/PadaukSgaw-bookbold.ttf"),
    ("PadaukSgaw.ttf", GH_BASE + "PadaukSgaw/PadaukSgaw.ttf"),
    ("PangLong_2011feb7.ttf", GH_BASE + "PangLong/PangLong_2011feb7.ttf"),
    ("PaohDragonApp.ttf", GH_BASE + "PaohDragonApp/PaohDragonApp.ttf"),
    ("Parabaik20080421.ttf", GH_BASE + "Parabaik/Parabaik20080421.ttf"),
    ("Parabaik_UTN11-3.ttf", GH_BASE + "Parabaik/Parabaik_UTN11-3.ttf"),
    ("pyidaungsu-1.2.ttf", GH_BASE + "Pyidaungsu/pyidaungsu-1.2.ttf"),
    ("pyidaungsu-1.3.ttf", GH_BASE + "Pyidaungsu/pyidaungsu-1.3.ttf"),
    ("pyidaungsu.ttf", GH_BASE + "Pyidaungsu/pyidaungsu.ttf"),
    ("Samsung_Myanmar.ttf", GH_BASE + "SamsungMyanmar/Samsung%20Myanmar.ttf"),
    ("Shan_Unicode.ttf", GH_BASE + "ShanUnicode/Shan%20Unicode.ttf"),
    ("ThanLwin.ttf", GH_BASE + "Thanlwin/ThanLwin.ttf"),
    ("TharLon.ttf", GH_BASE + "Tharlon/TharLon.ttf"),
    ("WinUniInnwa.ttf", GH_BASE + "WinUniInnwa/WinUniInnwa.ttf"),
    ("YoeYar-One_Bold.ttf", GH_BASE + "YoeYarOne/YoeYar-One%20Bold.ttf"),
    ("YoeYar-One_Light.ttf", GH_BASE + "YoeYarOne/YoeYar-One%20Light.ttf"),
    ("YoeYar-One_Regular.ttf", GH_BASE + "YoeYarOne/YoeYar-One%20Regular.ttf"),
    ("Yunghkio.ttf", GH_BASE + "Yunghkio/Yunghkio.ttf"),
    ("NotoSansMyanmar-Thin.ttf", GF_BASE + "notosansmyanmar/NotoSansMyanmar-Thin.ttf"),
    ("NotoSansMyanmar-ExtraLight.ttf", GF_BASE + "notosansmyanmar/NotoSansMyanmar-ExtraLight.ttf"),
    ("NotoSansMyanmar-Light.ttf", GF_BASE + "notosansmyanmar/NotoSansMyanmar-Light.ttf"),
    ("NotoSansMyanmar-Medium.ttf", GF_BASE + "notosansmyanmar/NotoSansMyanmar-Medium.ttf"),
    ("NotoSansMyanmar-SemiBold.ttf", GF_BASE + "notosansmyanmar/NotoSansMyanmar-SemiBold.ttf"),
    ("NotoSansMyanmar-ExtraBold.ttf", GF_BASE + "notosansmyanmar/NotoSansMyanmar-ExtraBold.ttf"),
    ("NotoSansMyanmar-Black.ttf", GF_BASE + "notosansmyanmar/NotoSansMyanmar-Black.ttf"),
    ("NotoSerifMyanmar-Thin.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-Thin.ttf"),
    ("NotoSerifMyanmar-ExtraLight.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-ExtraLight.ttf"),
    ("NotoSerifMyanmar-Light.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-Light.ttf"),
    ("NotoSerifMyanmar-Regular.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-Regular.ttf"),
    ("NotoSerifMyanmar-Medium.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-Medium.ttf"),
    ("NotoSerifMyanmar-SemiBold.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-SemiBold.ttf"),
    ("NotoSerifMyanmar-Bold.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-Bold.ttf"),
    ("NotoSerifMyanmar-ExtraBold.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-ExtraBold.ttf"),
    ("NotoSerifMyanmar-Black.ttf", GF_BASE + "notoserifmyanmar/NotoSerifMyanmar-Black.ttf")
]

print(f"Step 1/2: Downloading {len(DIRECT_FONT_LIST)} core & Noto multi-weight Unicode fonts...")
for local_name, url in tqdm(DIRECT_FONT_LIST, desc="Core & Noto Fonts"):
    dest = os.path.join(FONTS_DIR, local_name)
    if not os.path.exists(dest) or os.path.getsize(dest) < 1000:
        try:
            urllib.request.urlretrieve(url, dest)
        except Exception:
            pass

# Step 2: Harvest extra Masterpiece, Khyay, Waso, Chatu, Sanpya, and Saturngod Unicode fonts from GitHub archives
EXTRA_FONT_ARCHIVES = [
    ("fonts-myanmar-extra", "https://codeload.github.com/lecramyajiv/fonts-myanmar-extra/zip/refs/heads/main"),
    ("saturngod-unicode-fonts", "https://codeload.github.com/saturngod/myanmar-unicode-fonts/zip/refs/heads/master")
]

EXCLUDED_SUBSTRINGS = ("zawgyi", "wininnwa", "padaukghost")

print("\nStep 2/2: Harvesting extra community & Masterpiece Unicode fonts from GitHub archives...")
for repo_label, zip_url in EXTRA_FONT_ARCHIVES:
    try:
        req = urllib.request.Request(zip_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            zbytes = resp.read()
        with zipfile.ZipFile(io.BytesIO(zbytes)) as zf:
            extracted_from_repo = 0
            for member in zf.namelist():
                lower_m = member.lower()
                if lower_m.endswith((".ttf", ".otf")) and not any(bad in lower_m for bad in EXCLUDED_SUBSTRINGS):
                    base_fname = os.path.basename(member).replace(" ", "_")
                    if not base_fname:
                        continue
                    dest_path = os.path.join(FONTS_DIR, base_fname)
                    if not os.path.exists(dest_path) or os.path.getsize(dest_path) < 1000:
                        font_data = zf.read(member)
                        if len(font_data) > 2000:
                            with open(dest_path, "wb") as out_f:
                                out_f.write(font_data)
                            extracted_from_repo += 1
            print(f"  • Harvested {extracted_from_repo} additional unique Unicode fonts from {repo_label}")
    except Exception as e:
        print(f"  • Note: Could not fetch archive {repo_label}: {e}")

available_fonts = sorted([f for f in os.listdir(FONTS_DIR) if f.lower().endswith((".ttf", ".otf"))])
print("=" * 60)
print(f"TOTAL VERIFIED MYANMAR UNICODE FONTS READY: {len(available_fonts)}")
print("=" * 60)

## 4. Text Sanitizer, Zawgyi Auto-Fixer & `#H1/#H2/#H3` Tag Chunker
- Automatically detects and converts any stray Zawgyi text to standard Myanmar Unicode (NFC).
- Strips non-Burmese noise, invisible characters, and stray brackets.
- Parses trailing `#H1`, `#H2`, `#H3` tags and chunks stories into balanced book pages (~1,050 characters per page).

In [ ]:
import re
import unicodedata
try:
    from para.convert import zg_to_unicode
    from para.detect import is_zawgyi
    HAS_PARA = True
except ImportError:
    HAS_PARA = False

def clean_and_normalize_burmese(raw_text):
    if not raw_text:
        return ""
    if HAS_PARA and is_zawgyi(raw_text):
        text = zg_to_unicode(raw_text)
    else:
        text = raw_text
        
    t = unicodedata.normalize("NFC", text)
    t = re.sub(r'\[|\]', '', t)
    t = re.sub(r'[\u200B-\u200D\uFEFF\u00A0]', ' ', t)
    t = re.sub(r'<[^>]+>', ' ', t)
    t = re.sub(r'https?://\S+|www\.\S+', '', t)
    
    t = re.sub(r'#H1\b', ' __TAG_H1__ ', t, flags=re.IGNORECASE)
    t = re.sub(r'#H2\b', ' __TAG_H2__ ', t, flags=re.IGNORECASE)
    t = re.sub(r'#H3\b', ' __TAG_H3__ ', t, flags=re.IGNORECASE)
    
    allowed_pattern = r'[^\u1000-\u109F0-9\s၊။,\.\-\(\)\"\'“”:\!\?%/_a-zA-Z]'
    t = re.sub(allowed_pattern, ' ', t)
    
    t = t.replace('__TAG_H1__', '#H1')
    t = t.replace('__TAG_H2__', '#H2')
    t = t.replace('__TAG_H3__', '#H3')
    
    t = re.sub(r'[ \t]+', ' ', t)
    t = re.sub(r'\n\s*\n', '\n\n', t)
    return t.strip()

def parse_line_element(raw_line):
    line = raw_line.strip()
    if not line:
        return None
    if re.search(r'#H1\b', line, re.IGNORECASE) or line.startswith('# '):
        clean_text = re.sub(r'#H1\b', '', line, flags=re.IGNORECASE).lstrip('#').strip()
        return {"type": "h1", "text": clean_text}
    if re.search(r'#H2\b', line, re.IGNORECASE) or line.startswith('## '):
        clean_text = re.sub(r'#H2\b', '', line, flags=re.IGNORECASE).lstrip('#').strip()
        return {"type": "h2", "text": clean_text}
    if re.search(r'#H3\b', line, re.IGNORECASE) or line.startswith('### '):
        clean_text = re.sub(r'#H3\b', '', line, flags=re.IGNORECASE).lstrip('#').strip()
        return {"type": "h3", "text": clean_text}
    return {"type": "p", "text": line}

def chunk_text_into_book_pages(raw_text, target_chars_per_page=TARGET_CHARS_PER_PAGE):
    sanitized_text = clean_and_normalize_burmese(raw_text)
    raw_lines = [l.strip() for l in sanitized_text.split("\n") if l.strip()]
    
    elements = []
    for l in raw_lines:
        el = parse_line_element(l)
        if el and el["text"]:
            elements.append(el)
            
    has_any_heading = any(e["type"] in ["h1", "h2", "h3"] for e in elements)
    if not has_any_heading and elements:
        elements[0]["type"] = "h1"
        
    pages = []
    current_page_elements = []
    current_char_count = 0
    
    for el in elements:
        el_len = len(el["text"])
        if el["type"] == "h1" and current_page_elements:
            pages.append(current_page_elements)
            current_page_elements = [el]
            current_char_count = el_len
            continue
        if current_char_count + el_len > target_chars_per_page and current_char_count > 350:
            pages.append(current_page_elements)
            current_page_elements = [el]
            current_char_count = el_len
        else:
            current_page_elements.append(el)
            current_char_count += el_len
            
    if current_page_elements:
        pages.append(current_page_elements)
    return pages

print("Text sanitizer and #H1/#H2/#H3 book chunker ready.")

## 5. Native Browser HarfBuzz (`mym2`) Renderer with Intra-Page Multi-Font Mixing
- Supports **intra-page font pairing**: on 50% of pages, headings (`#H1`, `#H2`, `#H3`) are rendered in a distinct display/bold font while body paragraphs use a contrasting book font.
- Uses **Headless Chrome / Microsoft Edge** with a local HTTP font server for **100% authentic OpenType `mym2` Burmese shaping** at **~0.1s per page**.

In [ ]:
import io
import random
import tempfile
import threading
import http.server
import socketserver
from PIL import Image, ImageEnhance
import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.edge.options import Options as EdgeOptions

def start_local_font_server(fonts_dir):
    handler = lambda *args, **kwargs: http.server.SimpleHTTPRequestHandler(*args, directory=os.path.abspath(fonts_dir), **kwargs)
    httpd = socketserver.TCPServer(("127.0.0.1", 0), handler)
    port = httpd.server_address[1]
    thread = threading.Thread(target=httpd.serve_forever, daemon=True)
    thread.start()
    return httpd, port

def create_headless_browser():
    try:
        opts = ChromeOptions()
        opts.add_argument("--headless=new")
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--force-device-scale-factor=2")
        opts.add_argument(f"--window-size={PAGE_WIDTH_PX},{PAGE_HEIGHT_PX}")
        return webdriver.Chrome(options=opts)
    except Exception:
        opts = EdgeOptions()
        opts.add_argument("--headless=new")
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--force-device-scale-factor=2")
        opts.add_argument(f"--window-size={PAGE_WIDTH_PX},{PAGE_HEIGHT_PX}")
        return webdriver.Edge(options=opts)

def apply_book_scan_effects(pil_img):
    angle = random.uniform(-0.5, 0.5)
    img = pil_img.rotate(angle, resample=Image.BICUBIC, expand=True, fillcolor="white").convert("RGB")
    
    w, _ = img.size
    arr = np.array(img, dtype=np.float32)
    
    if random.random() < 0.35:
        shadow_side = random.choice(["left", "right"])
        gradient = np.linspace(0.85, 1.0, int(w * 0.20))
        if shadow_side == "right":
            gradient = gradient[::-1]
        if shadow_side == "left":
            arr[:, :len(gradient), :] *= gradient[np.newaxis, :, np.newaxis]
        else:
            arr[:, -len(gradient):, :] *= gradient[np.newaxis, :, np.newaxis]
            
    noise = np.random.normal(0, random.uniform(1.5, 3.2), arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)
    enhancer = ImageEnhance.Contrast(img)
    return enhancer.enhance(random.uniform(0.97, 1.03))

def inject_inline_styles(text_sentence):
    words = text_sentence.split()
    if len(words) < 5 or random.random() > 0.40:
        return text_sentence, text_sentence
        
    effect = random.choice(["bold", "italic", "underline", "quotes"])
    mid = len(words) // 2
    phrase = " ".join(words[mid-1:mid+2])
    
    if effect == "bold":
        html_sub = f"<b>{phrase}</b>"
        md_sub = f"**{phrase}**"
    elif effect == "italic":
        html_sub = f"<i>{phrase}</i>"
        md_sub = f"*{phrase}*"
    elif effect == "underline":
        html_sub = f"<u>{phrase}</u>"
        md_sub = f"<u>{phrase}</u>"
    else:
        html_sub = f"“{phrase}”"
        md_sub = f"“{phrase}”"
        
    html_out = text_sentence.replace(phrase, html_sub, 1)
    md_out = text_sentence.replace(phrase, md_sub, 1)
    return html_out, md_out

class BurmeseBrowserRenderer:
    def __init__(self, fonts_dir=FONTS_DIR):
        self.fonts_dir = fonts_dir
        self.httpd, self.port = start_local_font_server(fonts_dir)
        self.driver = create_headless_browser()
        self.driver.get(f"http://127.0.0.1:{self.port}/")
        
        excluded = {"zawgyi-one.ttf", "wininnwa.ttf", "padauk-zawgyi.ttf", "padaukghost.ttf"}
        self.profiles = []
        font_faces = []
        for idx, fname in enumerate(sorted(os.listdir(fonts_dir))):
            if fname.lower().endswith((".ttf", ".otf")) and fname.lower() not in excluded:
                clean_id = re.sub(r'[^a-zA-Z0-9]', '', os.path.splitext(fname)[0])
                fam_name = f"MMFont_{idx}_{clean_id}"
                self.profiles.append({"name": os.path.splitext(fname)[0], "family": fam_name})
                font_faces.append(f'@font-face {{ font-family: "{fam_name}"; src: url("http://127.0.0.1:{self.port}/{fname}") format("truetype"); }}')
                
        head_html = f'<meta charset="utf-8"><style>{"\n".join(font_faces)} html, body {{ margin: 0; width: {PAGE_WIDTH_PX}px; height: {PAGE_HEIGHT_PX}px; overflow: hidden; }} body {{ box-sizing: border-box; }}</style>'
        self.driver.execute_script("document.head.innerHTML = arguments[0];", head_html)
        print(f"Browser HarfBuzz renderer initialized with {len(self.profiles)} Myanmar Unicode fonts on port {self.port}.")

    def render_page(self, page_elements, output_png_path):
        body_profile = random.choice(self.profiles)
        # 50% of pages use a distinct header font paired with the body font
        header_profile = random.choice(self.profiles) if random.random() < 0.50 else body_profile
        
        body_fam = body_profile["family"]
        header_fam = header_profile["family"]
        
        h1_align = random.choice(["center", "left"])
        title_size = random.choice([22, 24, 26])
        h2_size = random.choice([18, 19, 20])
        h3_size = random.choice([16, 17])
        body_size = random.choice([14, 15, 16])
        line_height = random.choice([1.85, 1.95, 2.05])
        bg_color = random.choice(["#ffffff", "#fcf8ee", "#f7f4eb"])
        pad_x = random.randint(52, 68)
        pad_y = random.randint(54, 70)
        
        html_blocks = [f'<div style="font-family: \'{body_fam}\', sans-serif; background-color: {bg_color}; color: #161616; padding: {pad_y}px {pad_x}px; min-height: {PAGE_HEIGHT_PX}px; box-sizing: border-box;">']
        md_blocks = []
        
        for el in page_elements:
            el_type = el["type"]
            raw_t = unicodedata.normalize("NFC", el["text"].strip())
            if el_type == "h1":
                html_blocks.append(f'<h1 style="font-family: \'{header_fam}\', sans-serif; font-size: {title_size}px; text-align: {h1_align}; font-weight: bold; margin: 0 0 18px 0; line-height: 1.4;">{raw_t}</h1>')
                md_blocks.append(f"# {raw_t}")
            elif el_type == "h2":
                html_blocks.append(f'<h2 style="font-family: \'{header_fam}\', sans-serif; font-size: {h2_size}px; font-weight: bold; margin: 18px 0 10px 0; border-bottom: 1.5px solid #bbb; padding-bottom: 4px;">{raw_t}</h2>')
                md_blocks.append(f"## {raw_t}")
            elif el_type == "h3":
                html_blocks.append(f'<h3 style="font-family: \'{header_fam}\', sans-serif; font-size: {h3_size}px; font-weight: bold; margin: 14px 0 8px 0;">{raw_t}</h3>')
                md_blocks.append(f"### {raw_t}")
            else:
                h_snippet, m_snippet = inject_inline_styles(raw_t)
                html_blocks.append(f'<p style="font-size: {body_size}px; line-height: {line_height}; text-align: justify; text-indent: 32px; margin: 0 0 14px 0;">{h_snippet}</p>')
                md_blocks.append(m_snippet)
                
        html_blocks.append("</div>")
        self.driver.execute_script("document.body.innerHTML = arguments[0];", "".join(html_blocks))
        self.driver.execute_async_script("const cb = arguments[arguments.length - 1]; document.fonts.ready.then(() => cb());")
        
        png_bytes = self.driver.get_screenshot_as_png()
        raw_img = Image.open(io.BytesIO(png_bytes))
        scanned_img = apply_book_scan_effects(raw_img)
        
        output_dir = os.path.dirname(output_png_path)
        os.makedirs(output_dir, exist_ok=True)
        fd, temp_output_path = tempfile.mkstemp(prefix=".page_", suffix=".png", dir=output_dir)
        os.close(fd)
        try:
            scanned_img.save(temp_output_path, format="PNG")
            os.replace(temp_output_path, output_png_path)
        finally:
            if os.path.exists(temp_output_path):
                os.remove(temp_output_path)
        
        font_label = body_profile["name"] if body_profile["name"] == header_profile["name"] else f"{header_profile['name']}+{body_profile['name']}"
        return {"font": font_label, "body_font": body_profile["name"], "ground_truth": "\n\n".join(md_blocks).strip()}

    def close(self):
        try:
            self.driver.quit()
        except Exception:
            pass
        try:
            self.httpd.shutdown()
        except Exception:
            pass

print("BurmeseBrowserRenderer with 140+ fonts and intra-page font mixing ready.")

## 6. Process All Stories Across All 140+ Fonts
Chunks every `.txt` file in `raw_texts/`, renders pages via the native Browser HarfBuzz engine, and writes `train.jsonl` and `val.jsonl`.

In [ ]:
import glob
import json
from tqdm import tqdm

txt_paths = sorted(glob.glob(os.path.join(RAW_TEXTS_DIR, "*.txt")))
assert len(txt_paths) > 0, f"Please place at least one .txt file in {RAW_TEXTS_DIR}"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
renderer = BurmeseBrowserRenderer(FONTS_DIR)
dataset_entries = []
page_counter = 0
font_counts = {}

try:
    for txt_path in txt_paths:
        with open(txt_path, "r", encoding="utf-8") as f:
            full_text = f.read()
        
        page_chunks = chunk_text_into_book_pages(full_text)
        print(f"File '{os.path.basename(txt_path)}' -> {len(page_chunks)} book page chunks.")
        
        for chunk_elements in tqdm(page_chunks, desc=f"Rendering {os.path.basename(txt_path)}"):
            page_counter += 1
            img_name = f"page_{page_counter:05d}.png"
            img_full_path = os.path.join(IMAGES_OUTPUT_DIR, img_name)
            
            meta = renderer.render_page(chunk_elements, img_full_path)
            font_counts[meta["body_font"]] = font_counts.get(meta["body_font"], 0) + 1
            
            dataset_entries.append({
                "messages": [
                    {"role": "user", "content": "<image>Multi page parsing."},
                    {"role": "assistant", "content": meta["ground_truth"]}
                ],
                "images": [f"images/{img_name}"],
                "font": meta["font"]
            })
finally:
    renderer.close()

random.shuffle(dataset_entries)
split_point = max(1, int(len(dataset_entries) * (1 - VALIDATION_FRACTION)))
train_data = dataset_entries[:split_point]
val_data = dataset_entries[split_point:] if len(dataset_entries) > 1 else dataset_entries

with open(os.path.join(OUTPUT_DIR, "train.jsonl"), "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps({"messages": item["messages"], "images": item["images"]}, ensure_ascii=False) + "\n")

with open(os.path.join(OUTPUT_DIR, "val.jsonl"), "w", encoding="utf-8") as f:
    for item in val_data:
        f.write(json.dumps({"messages": item["messages"], "images": item["images"]}, ensure_ascii=False) + "\n")

print("\n" + "=" * 55)
print(f"Total Properly Shaped Book Pages Generated: {page_counter}")
print(f"Unique Body Fonts Sampled: {len(font_counts)}")
print(f"Train Split: {len(train_data)} | Validation Split: {len(val_data)}")
print("=" * 55)

## 7. Preview Rendered Book Pages
Verify that vowels (`ေ`), medials (`ြ`), and stacked consonants are 100% properly shaped.

In [ ]:
from IPython.display import display

for i in range(min(4, len(dataset_entries))):
    sample_img_path = os.path.join(OUTPUT_DIR, dataset_entries[i]["images"][0])
    print(f"\nPreview Page #{i+1} (Font Pairing: {dataset_entries[i]['font']}): {sample_img_path}")
    print("Assistant Ground Truth (first 160 chars):")
    print(dataset_entries[i]["messages"][1]["content"][:160] + "...")
    with Image.open(sample_img_path) as im:
        preview_im = im.copy()
        preview_im.thumbnail((450, 650))
        display(preview_im)

## 8. Package Dataset for Manual Upload
Creates a local ZIP archive of the generated dataset. Upload the files to GitHub, Hugging Face, or Kaggle yourself; this notebook does not access or publish to those services.

In [ ]:
import shutil

zip_path = shutil.make_archive(ZIP_NAME, "zip", OUTPUT_DIR)
print(f"Dataset archive ready for manual upload: {zip_path} ({os.path.getsize(zip_path) / (1024*1024):.2f} MB)")